In [2]:
# Cell 1 — Import Libraries
import os
import cv2
import glob
import random
import shutil
import numpy as np
from pathlib import Path
from collections import defaultdict
import albumentations as A

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [3]:
# Cell 2 — Define Paths & Settings
output_path = "../output/final_dataset"
FINAL_CLASSES = ['Plastic', 'Glass', 'Metal', 'Paper', 'Organic', 'Non-Recyclable']

TARGET = 108321  # هنوصل بيه كل class

# كمية الـ Augmentation المطلوبة لكل class
current_counts = {
    'Plastic':         108321,
    'Glass':            11203,
    'Metal':             9695,
    'Paper':            12292,
    'Organic':           6130,
    'Non-Recyclable':     705,
}

print("📊 الـ Augmentation المطلوبة:\n")
for cls, count in current_counts.items():
    needed = max(0, TARGET - count)
    print(f"  {cls:<20} الحالي: {count:>8} | محتاج: {needed:>8}")

📊 الـ Augmentation المطلوبة:

  Plastic              الحالي:   108321 | محتاج:        0
  Glass                الحالي:    11203 | محتاج:    97118
  Metal                الحالي:     9695 | محتاج:    98626
  Paper                الحالي:    12292 | محتاج:    96029
  Organic              الحالي:     6130 | محتاج:   102191
  Non-Recyclable       الحالي:      705 | محتاج:   107616


In [4]:
# Cell 3 — Define Augmentation Pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.4),
    A.GaussNoise(p=0.3),
    A.Blur(blur_limit=3, p=0.3),
    A.CLAHE(p=0.3),
    A.RandomScale(scale_limit=0.2, p=0.4),
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3
))

print("✅ Augmentation Pipeline defined:")
print("  - HorizontalFlip")
print("  - VerticalFlip")
print("  - RandomRotate90")
print("  - RandomBrightnessContrast")
print("  - HueSaturationValue")
print("  - GaussNoise")
print("  - Blur")
print("  - CLAHE")
print("  - RandomScale")

✅ Augmentation Pipeline defined:
  - HorizontalFlip
  - VerticalFlip
  - RandomRotate90
  - RandomBrightnessContrast
  - HueSaturationValue
  - GaussNoise
  - Blur
  - CLAHE
  - RandomScale


In [7]:
# Cell 4 — Helper Functions
def get_images_by_class(output_path, class_id):
    """جيب كل الصور اللي فيها class معين"""
    result = []
    for split in ['train', 'val', 'test']:
        label_files = glob.glob(f"{output_path}/{split}/labels/*.txt")
        for label_file in label_files:
            with open(label_file, 'r') as f:
                lines = f.readlines()
            for line in lines:
                if line.strip() and int(float(line.split()[0])) == class_id:
                    img_stem = Path(label_file).stem
                    for ext in ['jpg', 'jpeg', 'png']:
                        img_path = f"{output_path}/{split}/images/{img_stem}.{ext}"
                        if os.path.exists(img_path):
                            result.append({
                                'img': img_path,
                                'label': label_file,
                                'split': split
                            })
                    break
    return result

def augment_image(img_path, label_path):
    """عمل augmentation لصورة وlabel"""
    img = cv2.imread(img_path)
    if img is None:
        return None, None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    bboxes = []
    class_labels = []
    
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            class_labels.append(int(float(parts[0])))
            bboxes.append([float(x) for x in parts[1:5]])
    
    if not bboxes:
        return None, None
    
    try:
        augmented = transform(image=img, bboxes=bboxes, class_labels=class_labels)
    except Exception:
        return None, None
    
    if not augmented['bboxes']:
        return None, None
    
    aug_img = cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR)
    aug_lines = []
    for cls, bbox in zip(augmented['class_labels'], augmented['bboxes']):
        aug_lines.append(f"{cls} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
    
    return aug_img, aug_lines

import glob
print("✅ Helper Functions defined")

✅ Helper Functions defined


In [8]:
# Cell 5 — Run Augmentation
import random
random.seed(42)
np.random.seed(42)

# Non-Recyclable هنوصلها لـ 6000 بس مش 108321
aug_targets = {
    'Glass':           TARGET,
    'Metal':           TARGET,
    'Paper':           TARGET,
    'Organic':         TARGET,
    'Non-Recyclable':  6000,
}

for cls_name, target in aug_targets.items():
    class_id = FINAL_CLASSES.index(cls_name)
    current = current_counts[cls_name]
    needed = target - current

    if needed <= 0:
        print(f"✅ {cls_name}: مش محتاج augmentation")
        continue

    print(f"\n🔄 {cls_name}: محتاج {needed:,} صورة زيادة...")

    # جيب الصور الأصلية
    source_images = get_images_by_class(output_path, class_id)
    if not source_images:
        print(f"  ❌ مش لاقي صور!")
        continue

    print(f"  📸 صور أصلية متاحة: {len(source_images)}")

    generated = 0
    attempts = 0
    max_attempts = needed * 3

    while generated < needed and attempts < max_attempts:
        attempts += 1
        sample = random.choice(source_images)

        aug_img, aug_lines = augment_image(sample['img'], sample['label'])
        if aug_img is None:
            continue

        # اسم جديد للملف
        new_name = f"aug_{cls_name}_{generated:06d}"

        # احفظ في train بس
        dst_img = os.path.join(output_path, 'train', 'images', f"{new_name}.jpg")
        dst_label = os.path.join(output_path, 'train', 'labels', f"{new_name}.txt")

        cv2.imwrite(dst_img, aug_img)
        with open(dst_label, 'w') as f:
            f.writelines(aug_lines)

        generated += 1

        if generated % 10000 == 0:
            print(f"  ✅ {generated:,} / {needed:,}")

    print(f"  ✅ {cls_name}: اتعملت {generated:,} صورة")

print("\n🎉 Augmentation خلصت!")


🔄 Glass: محتاج 97,118 صورة زيادة...
  📸 صور أصلية متاحة: 101365
  ✅ 10,000 / 97,118
  ✅ 20,000 / 97,118
  ✅ 30,000 / 97,118
  ✅ 40,000 / 97,118
  ✅ 50,000 / 97,118
  ✅ 60,000 / 97,118
  ✅ 70,000 / 97,118
  ✅ 80,000 / 97,118
  ✅ 90,000 / 97,118
  ✅ Glass: اتعملت 97,118 صورة

🔄 Metal: محتاج 98,626 صورة زيادة...
  📸 صور أصلية متاحة: 17625
  ✅ 10,000 / 98,626
  ✅ 20,000 / 98,626
  ✅ 30,000 / 98,626
  ✅ 40,000 / 98,626
  ✅ 50,000 / 98,626
  ✅ 60,000 / 98,626
  ✅ 70,000 / 98,626
  ✅ 80,000 / 98,626
  ✅ 90,000 / 98,626
  ✅ Metal: اتعملت 98,626 صورة

🔄 Paper: محتاج 96,029 صورة زيادة...
  📸 صور أصلية متاحة: 9230
  ✅ 10,000 / 96,029
  ✅ 20,000 / 96,029
  ✅ 30,000 / 96,029
  ✅ 40,000 / 96,029
  ✅ 50,000 / 96,029
  ✅ 60,000 / 96,029
  ✅ 70,000 / 96,029
  ✅ 80,000 / 96,029
  ✅ 90,000 / 96,029
  ✅ Paper: اتعملت 96,029 صورة

🔄 Organic: محتاج 102,191 صورة زيادة...
  📸 صور أصلية متاحة: 2346
  ✅ 10,000 / 102,191
  ✅ 20,000 / 102,191
  ✅ 30,000 / 102,191
  ✅ 40,000 / 102,191
  ✅ 50,000 / 102,191
  ✅ 60,

In [10]:
# Cell 6 — Fast Final Verification
print("📊 الإحصائيات النهائية:\n")

for split in ['train', 'val', 'test']:
    images = glob.glob(f"{output_path}/{split}/images/*")
    labels = glob.glob(f"{output_path}/{split}/labels/*")
    print(f"  {split}: {len(images)} صورة, {len(labels)} label")

total_imgs = sum(len(glob.glob(f"{output_path}/{split}/images/*")) for split in ['train','val','test'])
print(f"\n  📊 إجمالي الصور: {total_imgs}")

# عد الـ augmented files بس
print("\n📊 الـ Augmented Files:\n")
for cls in FINAL_CLASSES:
    aug_files = glob.glob(f"{output_path}/train/images/aug_{cls}_*")
    print(f"  {cls:<20} aug: {len(aug_files)}")

📊 الإحصائيات النهائية:

  train: 424279 صورة, 424279 label
  val: 6572 صورة, 6572 label
  test: 6475 صورة, 6475 label

  📊 إجمالي الصور: 437326

📊 الـ Augmented Files:

  Plastic              aug: 0
  Glass                aug: 97118
  Metal                aug: 98626
  Paper                aug: 96029
  Organic              aug: 102191
  Non-Recyclable       aug: 5295
